In [ ]:
from keras.utils import load_img, img_to_array
from keras.models import load_model
from zipfile import ZipFile
import numpy as np
import os

In [ ]:
# functions
def unzip_testset():
    CWD = os.getcwd()
    with ZipFile(os.path.join(CWD,r'data\\test-imgs.zip')) as TestImgFile:
        TestImgFile.extractall(os.path.join(CWD,r'data\\test-imgs'))

def load_images_for_prediction(img_folder:str, target_size:tuple):
    valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    image_paths = []

    for root, _, files in os.walk(img_folder):
        for name in sorted(files):
            if name.lower().endswith(valid_ext):
                image_paths.append(os.path.join(root, name))

    if not image_paths:
        raise ValueError(f"No valid images found in: {img_folder}")

    images = []
    for path in image_paths:
        img = load_img(path, target_size=target_size, color_mode="rgb")
        arr = img_to_array(img).astype("float32") / 255.0
        images.append(arr)

    batch = np.stack(images, axis=0)
    return batch, image_paths

# unzip dataset
if not os.path.exists(os.path.join(os.getcwd(),r'data\\test-imgs')): 
    IMG_FOLDER = os.path.join(os.getcwd(),r'data\\test-imgs')
    unzip_testset()
else: 
    IMG_FOLDER = os.path.join(os.getcwd(),r'data\\test-imgs')
    print('File already exists.')

File already exists.


In [ ]:
MODEL_PATH = r'models/model.keras'  # change if needed
FIXED_MODEL_PATH = r'models/model_fixed.keras'
INPUT_SHAPE = (224, 224)

model_file = os.path.join(os.getcwd(), MODEL_PATH)
fixed_model_file = os.path.join(os.getcwd(), FIXED_MODEL_PATH)


def _build_model_for_legacy_weights():
    from keras import layers
    from keras.models import Sequential
    from keras.applications import MobileNetV2

    base_model = MobileNetV2(weights=None, include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    rebuilt_model = Sequential([
        base_model,
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return rebuilt_model


preferred_path = fixed_model_file if os.path.exists(fixed_model_file) else model_file
try:
    model = load_model(preferred_path, compile=False)
    print(f'Loaded model from: {preferred_path}')
except Exception as e:
    print(f'Direct load failed ({e.__class__.__name__}). Converting legacy model...')
    import tempfile

    model = _build_model_for_legacy_weights()
    with tempfile.TemporaryDirectory() as tmpdir:
        with ZipFile(model_file) as zf:
            zf.extract('model.weights.h5', path=tmpdir)
        model.load_weights(os.path.join(tmpdir, 'model.weights.h5'))

    model.save(fixed_model_file)
    print(f'Saved converted model to: {fixed_model_file}')

X, image_paths = load_images_for_prediction(IMG_FOLDER, INPUT_SHAPE)
predictions = model.predict(X, verbose=0)

print('Batch shape:', X.shape)
print('Predictions shape:', predictions.shape)
predictions[:3]



In [ ]:
import math
import matplotlib.pyplot as plt

# Optional: set class names in model index order, e.g. ['cat', ...]
CLASS_NAMES = None

def show_predictions_grid(images, preds, paths, class_names=None, cols=4):
    n = len(images)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    pred_idx = np.argmax(preds, axis=1)
    pred_conf = np.max(preds, axis=1)

    for i in range(n):
        axes[i].imshow(images[i])
        label = class_names[pred_idx[i]] if class_names is not None else f'Class {pred_idx[i]}'
        file_name = os.path.basename(paths[i])
        axes[i].set_title(f'{file_name}\nPred: {label} ({pred_conf[i]:.2%})')
        axes[i].axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

show_predictions_grid(X, predictions, image_paths, class_names=CLASS_NAMES, cols=4)
